# 02 — Reversals and Excursions in a Bistable Toy Model

This notebook develops the standard **double-well interpretation** of geomagnetic reversals.  
It is aligned with the repository folder `models/bistable_models/`.

## Learning goals

- understand why bistability is a natural reduced picture for polarity switching,
- simulate a noisy double-well system,
- distinguish full reversals from small fluctuations,
- compute reversal statistics and visualize phase-space structure.


In [ ]:

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 4.5)
plt.rcParams["axes.grid"] = True

ROOT = Path.cwd()
if not (ROOT / "models").exists() and (ROOT.parent / "models").exists():
    ROOT = ROOT.parent

print("Working directory:", Path.cwd())
print("Repository root guessed as:", ROOT)

def repo_file_info(relpath):
    p = ROOT / relpath
    return {"exists": p.exists(), "size": p.stat().st_size if p.exists() else None, "path": str(p)}

def sign_changes(x):
    s = np.sign(x)
    s[s == 0] = np.nan
    valid = ~np.isnan(s)
    sv = s[valid]
    return int(np.sum(sv[1:] * sv[:-1] < 0))

def polarity_series(x):
    p = np.sign(x)
    if len(p) == 0:
        return p
    # carry last sign through zeros
    for i in range(1, len(p)):
        if p[i] == 0:
            p[i] = p[i-1]
    if p[0] == 0:
        nz = np.flatnonzero(p != 0)
        if len(nz):
            p[:nz[0]] = p[nz[0]]
    return p

def residence_times_from_signal(x, dt=1.0):
    p = polarity_series(np.asarray(x))
    if len(p) == 0:
        return np.array([])
    durations = []
    current = p[0]
    count = 1
    for val in p[1:]:
        if val == current:
            count += 1
        else:
            durations.append(count * dt)
            current = val
            count = 1
    durations.append(count * dt)
    return np.array(durations)

def power_spectrum(x, dt=1.0):
    x = np.asarray(x)
    x = x - np.mean(x)
    freqs = np.fft.rfftfreq(len(x), d=dt)
    spec = np.abs(np.fft.rfft(x))**2 / len(x)
    return freqs[1:], spec[1:]

rng = np.random.default_rng(42)


## 1. Bistability as a reversal mechanism

A standard reduced equation is

$$
dx = (ax - bx^3)\,dt + \sigma\,dW_t,
$$

with \(a>0\), \(b>0\).  
The deterministic part derives from the effective potential

$$
U(x) = -\frac{a}{2}x^2 + \frac{b}{4}x^4.
$$

This gives two preferred states near

$$
x_\star = \pm \sqrt{\frac{a}{b}}.
$$

These two states are interpreted as opposite dipole polarities.


In [ ]:

def simulate_double_well(n_steps=50000, dt=0.01, a=1.0, b=1.0, sigma=0.55, seed=42):
    rng = np.random.default_rng(seed)
    x = np.zeros(n_steps)
    for i in range(1, n_steps):
        drift = a * x[i-1] - b * x[i-1]**3
        x[i] = x[i-1] + drift * dt + sigma * np.sqrt(dt) * rng.standard_normal()
    t = np.arange(n_steps) * dt
    return t, x

t, x = simulate_double_well()
fig, ax = plt.subplots()
ax.plot(t, x, lw=0.8)
ax.set_title("Bistable dipole proxy")
ax.set_xlabel("time")
ax.set_ylabel("x(t)")
plt.show()


## 2. The effective potential

The next figure shows why the model has two preferred states.


In [ ]:

a, b = 1.0, 1.0
xx = np.linspace(-2, 2, 500)
U = -(a/2)*xx**2 + (b/4)*xx**4

fig, ax = plt.subplots()
ax.plot(xx, U, lw=2)
ax.set_title("Double-well potential")
ax.set_xlabel("x")
ax.set_ylabel("U(x)")
plt.show()


## 3. Reversal counting

We now compute simple reversal diagnostics based on zero crossings of the dipole proxy, with a mild persistence filter.


In [ ]:

def reversal_times(x, dt=1.0, persistence=5):
    p = polarity_series(x)
    times = []
    for i in range(1, len(p)-persistence):
        if p[i-1] != p[i] and np.all(p[i:i+persistence] == p[i]):
            times.append(i * dt)
    return np.array(times)

rev_t = reversal_times(x, dt=t[1]-t[0], persistence=20)
res = residence_times_from_signal(x, dt=t[1]-t[0])

print("Estimated reversals:", len(rev_t))
print("Mean residence time:", res.mean())

fig, axes = plt.subplots(2, 1, figsize=(10, 8))
axes[0].plot(t, x, lw=0.8)
for rt in rev_t:
    axes[0].axvline(rt, ls="--", lw=0.8)
axes[0].set_title("Reversal candidates")
axes[0].set_xlabel("time")
axes[0].set_ylabel("x(t)")

axes[1].hist(res, bins=30)
axes[1].set_title("Residence-time distribution")
axes[1].set_xlabel("duration")
axes[1].set_ylabel("count")

plt.tight_layout()
plt.show()


## 4. Excursions versus full reversals

Not every visit near zero becomes a full reversal.  
The following diagnostic highlights low-amplitude episodes that may be interpreted as excursion-like events.


In [ ]:

threshold = 0.3
low_dipole = np.abs(x) < threshold

fig, ax = plt.subplots()
ax.plot(t, x, lw=0.8, label="dipole proxy")
ax.fill_between(t, -threshold, threshold, where=low_dipole, alpha=0.2, label="low-dipole zone")
ax.axhline(threshold, ls="--", lw=1)
ax.axhline(-threshold, ls="--", lw=1)
ax.set_title("Low-dipole episodes and possible excursions")
ax.set_xlabel("time")
ax.set_ylabel("x(t)")
ax.legend()
plt.show()


## 5. Parameter scan

A professional notebook should show mechanism sensitivity.  
Here we vary the noise amplitude and compare reversal counts.


In [ ]:

sigmas = np.linspace(0.2, 0.9, 8)
counts = []

for s in sigmas:
    _, xx = simulate_double_well(sigma=float(s), seed=42)
    counts.append(len(reversal_times(xx, dt=0.01, persistence=20)))

fig, ax = plt.subplots()
ax.plot(sigmas, counts, marker="o")
ax.set_title("Estimated reversal count versus noise amplitude")
ax.set_xlabel("sigma")
ax.set_ylabel("number of reversals")
plt.show()


## 6. Connection to the repository

This notebook corresponds conceptually to:

- `models/bistable_models/double_well.py`
- `models/bistable_models/stochastic_forcing.py`
- `diagnostics/polarity.py`
- `diagnostics/reversal_statistics.py`

If the repository scripts are later populated with functions, the numerical experiments in this notebook can be refactored to call them directly.


## 7. Suggested exercises

1. Change the persistence threshold and observe how reversal counts change.  
2. Change `a` and `b` and relate the result to barrier height.  
3. Estimate the mean waiting time as a function of `sigma`.  
4. Compare this notebook’s results with the domino-model notebook when available.
